## Install and import prefered Libraris and Load & Read dataset

In [ ]:
pip install pandas numpy jupyter ipykernel

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import difflib
import json
from datetime import datetime

pd.set_option('display.max_columns', None)

RAW_PATH = "Dataset/Titanic_Dataset.csv"
df_raw = pd.read_csv(RAW_PATH)

print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
df_raw.head()

Rows: 891, Columns: 12


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Completeness & Structural QA

First automation: a reusable function that profiles **any** dataframe - completeness %, dtype, and unique-value counts per column. This is the kind of QA script that can run on every new field-data batch to flag issues before they reach production.

In [ ]:
def completeness_report(df: pd.DataFrame) -> pd.DataFrame:
    '''Column-level QA profile: completeness %, dtype, uniqueness.'''
    total = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isnull().sum(),
        "missing_pct": (df.isnull().sum() / total * 100).round(2),
        "unique_values": df.nunique(),
    })
    report["complete_pct"] = (100 - report["missing_pct"]).round(2)
    return report.sort_values("missing_pct", ascending=False)

report_before = completeness_report(df_raw)
report_before

,dtype,missing_count,missing_pct,unique_values,complete_pct
Cabin,str,687,77.10,147,22.90
Age,float64,177,19.87,88,80.13
Embarked,str,2,0.22,3,99.78
PassengerId,int64,0,0.00,891,100.00
Name,str,0,0.00,891,100.00
Pclass,int64,0,0.00,3,100.00
Survived,int64,0,0.00,2,100.00
Sex,str,0,0.00,2,100.00
Parch,int64,0,0.00,7,100.00
SibSp,int64,0,0.00,7,100.00


In [ ]:
overall_completeness = 100 - (df_raw.isnull().sum().sum() / df_raw.size * 100)
print(f"Overall dataset completeness: {overall_completeness:.2f}%")
print(f"Columns with missing data: {list(report_before[report_before['missing_count'] > 0].index)}")

Overall dataset completeness: 91.90%
Columns with missing data: ['Cabin', 'Age', 'Embarked']


## 2. Duplicate Detection - Exact + Near-Duplicate

Real-world field data rarely has *exact* duplicate rows - it has **near-duplicates**: the same record entered twice with slightly different spelling, spacing, or formatting (e.g. the same POI submitted by two field agents as "Cafe Aroma" and "Cafe  Aroma "). Exact-match `duplicated()` misses these. Below: exact-match check + a fuzzy-matching pass using string similarity, the same technique used to catch duplicate POI/business-name entries.

In [ ]:
exact_dupes = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

def find_near_duplicates(series: pd.Series, threshold: float = 0.92):
    
    values = series.dropna().astype(str).str.strip().str.lower().unique()
    pairs = []
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            ratio = difflib.SequenceMatcher(None, values[i], values[j]).ratio()
            if ratio >= threshold:
                pairs.append((values[i], values[j], round(ratio, 3)))
    return pd.DataFrame(pairs, columns=["value_a", "value_b", "similarity"])

near_dupe_names = find_near_duplicates(df_raw["Name"], threshold=0.90)
print(f"Potential near-duplicate 'Name' entries found: {len(near_dupe_names)}")
near_dupe_names.head(10)

Exact duplicate rows: 0


Potential near-duplicate 'Name' entries found: 5


,value_a,value_b,similarity
0,"williams, mr. charles eugene","williams, mr. charles duane",0.909
1,"bing, mr. lee","ling, mr. lee",0.923
2,"youseff, mr. gerious","yousseff, mr. gerious",0.976
3,"healy, miss. hanora ""nora""","hegarty, miss. hanora ""nora""",0.926
4,"thayer, mr. john borland jr","thayer, mr. john borland",0.941


## 3. Formatting Standardization

Field-collected data commonly has inconsistent casing, stray whitespace, and mixed category labels (e.g. "Male"/"male"/"M"). This step standardizes categorical/text fields and logs every change made - an audit trail is essential for QA sign-off.

In [ ]:
df_clean = df_raw.copy()
change_log = []

def standardize_text_column(df, col, log):
    before = df[col].copy()
    df[col] = df[col].astype(str).str.strip()
    if df[col].str.contains(r"[A-Za-z]", na=False).any() and col in ["Sex", "Embarked"]:
        df[col] = df[col].str.lower()
    changed = (before.astype(str) != df[col]).sum()
    if changed:
        log.append({"column": col, "change_type": "whitespace/case standardization", "rows_affected": int(changed)})
    return df

for col in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:
    df_clean = standardize_text_column(df_clean, col, change_log)

# Extract a structured sub-field from free text — same pattern used to pull
# a "category" or "sub-type" out of a raw POI name/description field
df_clean["Title"] = df_clean["Name"].str.extract(r",\s*([^\.]+)\.")
change_log.append({"column": "Title (new)", "change_type": "extracted structured field from Name", "rows_affected": int(df_clean["Title"].notna().sum())})

pd.DataFrame(change_log)

,column,change_type,rows_affected
0,Name,whitespace/case standardization,2
1,Cabin,whitespace/case standardization,687
2,Embarked,whitespace/case standardization,891
3,Title (new),extracted structured field from Name,891


## 4. Rule-Based Missing-Value Handling (with audit flags)

Missing values are **never silently filled** - each imputation is documented and a boolean flag column marks which rows were touched, so downstream reviewers/field teams know exactly what was inferred vs. originally collected.

In [ ]:
imputation_log = []

df_clean["Age_was_missing"] = df_clean["Age"].isnull()
age_median_by_class = df_clean.groupby("Pclass")["Age"].transform("median")
rows_before = df_clean["Age"].isnull().sum()
df_clean["Age"] = df_clean["Age"].fillna(age_median_by_class)
imputation_log.append({"column": "Age", "rule": "median age per Pclass", "rows_imputed": int(rows_before)})

df_clean["Embarked_was_missing"] = df_clean["Embarked"].isnull() | (df_clean["Embarked"] == "nan")
mode_embarked = df_clean["Embarked"].mode(dropna=True)[0]
rows_before = df_clean["Embarked_was_missing"].sum()
df_clean.loc[df_clean["Embarked_was_missing"], "Embarked"] = mode_embarked
imputation_log.append({"column": "Embarked", "rule": f"mode fill ('{mode_embarked}')", "rows_imputed": int(rows_before)})

df_clean["Cabin_known"] = df_clean["Cabin"].notna() & (df_clean["Cabin"] != "nan")
df_clean["Cabin"] = df_clean["Cabin"].where(df_clean["Cabin_known"], "unknown")
imputation_log.append({"column": "Cabin", "rule": "flagged as 'unknown' (too sparse to impute — 77% missing)", "rows_imputed": int((~df_clean['Cabin_known']).sum())})

pd.DataFrame(imputation_log)

,column,rule,rows_imputed
0,Age,median age per Pclass,177
1,Embarked,mode fill ('s'),2
2,Cabin,flagged as 'unknown' (too sparse to impute — 7...,687


## 5. Automated QA Report (exportable)

A single function producing a stakeholder-ready QA summary - the kind of report requested in the JD ("Prepare reports on data quality, field progress, and errors found"). Exports to both JSON (for pipelines) and CSV (for sharing in Excel/Sheets).

In [ ]:
def generate_qa_report(df_before, df_after, near_dupes_df, imputation_log):
    report = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "total_records": len(df_after),
        "completeness_before_pct": round(100 - (df_before.isnull().sum().sum() / df_before.size * 100), 2),
        "completeness_after_pct": round(100 - (df_after.isnull().sum().sum() / df_after.size * 100), 2),
        "exact_duplicates_found": int(df_before.duplicated().sum()),
        "near_duplicate_pairs_flagged": len(near_dupes_df),
        "fields_imputed": imputation_log,
        "records_with_imputed_age": int(df_after["Age_was_missing"].sum()),
        "records_with_imputed_embarked": int(df_after["Embarked_was_missing"].sum()),
        "records_missing_cabin": int((~df_after["Cabin_known"]).sum()),
    }
    return report

qa_report = generate_qa_report(df_raw, df_clean, near_dupe_names, imputation_log)

with open("qa_report.json", "w") as f:
    json.dump(qa_report, f, indent=2)

pd.json_normalize(qa_report).T.rename(columns={0: "value"})

,value
generated_at,2026-07-13T15:50:27
total_records,891
completeness_before_pct,91.9
completeness_after_pct,100.0
exact_duplicates_found,0
near_duplicate_pairs_flagged,5
fields_imputed,"[{'column': 'Age', 'rule': 'median age per Pcl..."
records_with_imputed_age,177
records_with_imputed_embarked,2
records_missing_cabin,687


In [ ]:
column_qa_after = completeness_report(df_clean[["Age", "Cabin", "Embarked", "Sex", "Name", "Ticket"]])
column_qa_after.to_csv("column_qa_report.csv")
column_qa_after

,dtype,missing_count,missing_pct,unique_values,complete_pct
Age,float64,0,0.0,88,100.0
Cabin,str,0,0.0,148,100.0
Embarked,str,0,0.0,3,100.0
Sex,str,0,0.0,2,100.0
Name,str,0,0.0,891,100.0
Ticket,str,0,0.0,681,100.0


## 6. Before vs. After - Data Quality Improvement

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Overall completeness (%)", "Missing Age", "Missing Embarked", "Missing/Unknown Cabin", "Exact duplicates", "Near-duplicate Name pairs flagged"],
    "Before": [
        round(100 - (df_raw.isnull().sum().sum() / df_raw.size * 100), 2),
        int(df_raw["Age"].isnull().sum()),
        int(df_raw["Embarked"].isnull().sum()),
        int(df_raw["Cabin"].isnull().sum()),
        int(df_raw.duplicated().sum()),
        len(near_dupe_names),
    ],
    "After": [
        round(100 - (df_clean.isnull().sum().sum() / df_clean.size * 100), 2),
        0,
        0,
        int((~df_clean["Cabin_known"]).sum()),
        int(df_clean.duplicated().sum()),
        "reviewed (flagged, not auto-merged)",
    ],
})
summary

,Metric,Before,After
0,Overall completeness (%),91.9,100.0
1,Missing Age,177.0,0
2,Missing Embarked,2.0,0
3,Missing/Unknown Cabin,687.0,687
4,Exact duplicates,0.0,0
5,Near-duplicate Name pairs flagged,5.0,"reviewed (flagged, not auto-merged)"


In [ ]:
df_clean.to_csv("cleaned_dataset.csv", index=False)
print("Saved: cleaned_dataset.csv, qa_report.json, column_qa_report.csv")

Saved: cleaned_dataset.csv, qa_report.json, column_qa_report.csv
